<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/FeedbackSummarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from transformers import pipeline, logging
logging.set_verbosity_error()
import pandas as pd

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class TextSummarizer:
    def __init__(self, model_name="sshleifer/distilbart-cnn-12-6"):
        """Initialize the summarizer with a pre-trained model.

        Args:
            model_name (str): Name of the pre-trained model to use.
        """
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.model.to(self.device)

    def summarize(self, text, max_length=130, min_length=30, length_penalty=2.0,
                  repetition_penalty=2.0, num_beams=4, early_stopping=True):
        """Generate a summary for the given text.

        Args:
            text (str): The text to summarize
            max_length (int): Maximum length of the summary
            min_length (int): Minimum length of the summary
            length_penalty (float): Penalty for longer summaries
            repetition_penalty (float): Penalty for repeated tokens
            num_beams (int): Number of beams for beam search
            early_stopping (bool): Whether to stop when all beams are finished

        Returns:
            str: The generated summary
        """
        try:
            # Tokenize the input text
            inputs = self.tokenizer(text, max_length=1024, truncation=True,
                                    padding="max_length", return_tensors="pt"
                                    ).to(self.device)
            # Generate summary
            summary_ids = self.model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=max_length,
                min_length=min_length,
                length_penalty=length_penalty,
                repetition_penalty=repetition_penalty,
                no_repeat_ngram_size=3,
                num_beams=num_beams,
                early_stopping=early_stopping
            )
            # Decode and return the summary
            summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            return summary
        except Exception as e:
            print(f"Error during summarization: {str(e)}")
            return text



In [6]:
# Initialize the basic summarizer
summarizer = TextSummarizer()

# Sample text to summarize
sample_text = """
One big learning in Markets

When markets become deeply oversold
Stocks get battered left right & center without much differentiation (like we saw in March end)

At that time if you are a long term investor, it usually makes sense to look at fundamentally strong companies which are severely beaten down because you get them at mouth watering valuations

Provided you understand the business model well & have been tracking the company for the last few quarters

High RS stocks can provide stability during corrections
but historically the strongest returns have mostly come from businesses that corrected sharply because of negative sentiment

We have seen this play out across cycles many times
"""

# Generate a summary
summary = summarizer.summarize(sample_text)
print("Basic Summary:\n", summary)
print(len(sample_text))
print(len(summary))


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Basic Summary:
  When markets become deeply oversold, stocks get battered left right & center without much differentiation . Historically the strongest returns have mostly come from businesses that corrected sharply because of negative sentiment . We have seen this play out across cycles many times many times .
698
296


In [16]:
# Test with shorter text
short_text = "AI is changing the world by automating tasks and providing insights from large datasets."
short_summary = summarizer.summarize(short_text)
print("Short Text Summary:\n", short_summary)

Short Text Summary:
  AI is changing the world by automating tasks and providing insights from large datasets . It's changing the way we look at our jobs in a new way to make sure we don't have to rely on relying on humans .


In [ ]:
from transformers import LEDForConditionalGeneration, LEDTokenizer

# 1. Initialize the correct model for summarization
tokenizer = LEDTokenizer.from_pretrained("allenai/led-base-16384")
model = LEDForConditionalGeneration.from_pretrained("allenai/led-base-16384")

# 2. Prepare text
text = """
One big learning in Markets

When markets become deeply oversold
Stocks get battered left right & center without much differentiation (like we saw in March end)

At that time if you are a long term investor, it usually makes sense to look at fundamentally strong companies which are severely beaten down because you get them at mouth watering valuations

Provided you understand the business model well & have been tracking the company for the last few quarters

High RS stocks can provide stability during corrections
but historically the strongest returns have mostly come from businesses that corrected sharply because of negative sentiment

We have seen this play out across cycles many times

"""
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=4096)

# 3. Generate summary IDs
# global_attention_mask is required for Longformer; usually set to the first <s> token
summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=128, early_stopping=True)

# 4. Decode and print
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)
print(len(text))
print(len(summary))